<img src="http://developer.download.nvidia.com/notebooks/dlsw-notebooks/tensorrt_torchtrt_efficientnet/nvidia_logo.png" width="90px">

# PySpark LLM Inference: MinerU-HTML Processing

In this notebook, we demonstrate distributed batch inference with [MinerU-HTML](https://github.com/opendatalab/MinerU-HTML/tree/main) framework and model.

In [1]:
import os

# Manually enable Huggingface tokenizer parallelism to avoid disabling with PySpark parallelism.
# See (https://github.com/huggingface/transformers/issues/5486) for more info. 
os.environ["TOKENIZERS_PARALLELISM"] = "true"

# vLLM does CUDA init at import time. Forking will try to re-initialize CUDA if vLLM was imported before and throw an error.
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"

In [2]:
from huggingface_hub import snapshot_download

MODEL_PATH = os.path.abspath("mineru-html")
snapshot_download(
    repo_id="opendatalab/MinerU-HTML",
    local_dir=MODEL_PATH
)

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

'/home/rishic/Code/myforks/spark-rapids-examples/examples/ML+DL-Examples/Spark-DL/dl_inference/mineru-html/mineru-html'

## Warmup: Running locally

**Note**: If the driver node does not have sufficient GPU capacity, proceed to the PySpark section.

In [3]:
from dripper.api import Dripper

dripper = Dripper(
    config={
        'model_path': MODEL_PATH,
        'vllm_kwargs': {
            'gpu_memory_utilization': 0.15,
        }
    }
)

In [4]:
html_content = """
<html>
  <body>
    <div>
    <h1>This is a title</h1>
    <p>This is a paragraph</p>
    <p>This is another paragraph</p>
    </div>
    <div>
    <p>Related content</p>
    <p>Advertising content</p>
    </div>
  </body>
</html>
"""

result = dripper.process(html_content)

[2025-12-09 15:08:28] INFO api.py:394: Starting to process 1 inputs
[2025-12-09 15:08:28] INFO api.py:208: Loading model: /home/rishic/Code/myforks/spark-rapids-examples/examples/ML+DL-Examples/Spark-DL/dl_inference/mineru-html/mineru-html


INFO 12-09 15:08:28 [utils.py:253] non-default args: {'gpu_memory_utilization': 0.15, 'disable_log_stats': True, 'model': '/home/rishic/Code/myforks/spark-rapids-examples/examples/ML+DL-Examples/Spark-DL/dl_inference/mineru-html/mineru-html'}
INFO 12-09 15:08:28 [model.py:631] Resolved architecture: Qwen3ForCausalLM
INFO 12-09 15:08:28 [model.py:1745] Using max model len 40960
INFO 12-09 15:08:28 [scheduler.py:216] Chunked prefill is enabled with max_num_batched_tokens=8192.
(EngineCore_DP0 pid=67839) INFO 12-09 15:08:30 [core.py:93] Initializing a V1 LLM engine (v0.11.1) with config: model='/home/rishic/Code/myforks/spark-rapids-examples/examples/ML+DL-Examples/Spark-DL/dl_inference/mineru-html/mineru-html', speculative_config=None, tokenizer='/home/rishic/Code/myforks/spark-rapids-examples/examples/ML+DL-Examples/Spark-DL/dl_inference/mineru-html/mineru-html', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:00<00:00,  7.35it/s]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:00<00:00,  7.34it/s]
(EngineCore_DP0 pid=67839) 


(EngineCore_DP0 pid=67839) INFO 12-09 15:08:32 [gpu_model_runner.py:3334] Model loading took 1.1201 GiB memory and 0.284737 seconds
(EngineCore_DP0 pid=67839) INFO 12-09 15:08:35 [backends.py:631] Using cache directory: /home/rishic/.cache/vllm/torch_compile_cache/0bb15e202f/rank_0_0/backbone for vLLM's torch.compile
(EngineCore_DP0 pid=67839) INFO 12-09 15:08:35 [backends.py:647] Dynamo bytecode transform time: 3.26 s
(EngineCore_DP0 pid=67839) INFO 12-09 15:08:37 [backends.py:210] Directly load the compiled graph(s) for dynamic shape from the cache, took 1.949 s
(EngineCore_DP0 pid=67839) INFO 12-09 15:08:38 [monitor.py:34] torch.compile takes 5.21 s in total
(EngineCore_DP0 pid=67839) INFO 12-09 15:08:38 [gpu_worker.py:359] Available KV cache memory: 4.55 GiB
(EngineCore_DP0 pid=67839) INFO 12-09 15:08:38 [kv_cache_utils.py:1229] GPU KV cache size: 42,624 tokens
(EngineCore_DP0 pid=67839) INFO 12-09 15:08:38 [kv_cache_utils.py:1234] Maximum concurrency for 40,960 tokens per request:

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 51/51 [00:00<00:00, 74.68it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 35/35 [00:00<00:00, 79.63it/s]


(EngineCore_DP0 pid=67839) INFO 12-09 15:08:40 [gpu_model_runner.py:4240] Graph capturing finished in 1 secs, took 0.50 GiB
(EngineCore_DP0 pid=67839) INFO 12-09 15:08:40 [core.py:250] init engine (profile, create kv cache, warmup model) took 8.02 seconds
INFO 12-09 15:08:40 [llm.py:352] Supported tasks: ['generate']


[2025-12-09 15:08:40] INFO api.py:214: Model loading completed
[2025-12-09 15:08:40] INFO api.py:412: Starting model inference


Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

[2025-12-09 15:08:40] INFO api.py:456: Processing completed, output 1 results


In [5]:
print(result[0].main_html)

<html><body>
<div>
<h1 _item_id="1">This is a title</h1>
<p _item_id="2">This is a paragraph</p>
<p _item_id="3">This is another paragraph</p>
</div>
</body></html>


Unload the model to free up the GPU for the PySpark section.

In [6]:
import contextlib
import gc
import torch
from vllm.distributed import destroy_model_parallel, destroy_distributed_environment

def cleanup():
    destroy_model_parallel()
    destroy_distributed_environment()
    with contextlib.suppress(AssertionError):
        torch.distributed.destroy_process_group()
    gc.collect()
    torch.cuda.empty_cache()

del dripper
cleanup()

[rank0]:[W1209 15:08:46.264724515 ProcessGroupNCCL.cpp:1524] Warning: WARNING: destroy_process_group() was not called before program exit, which can leak resources. For more info, please see https://pytorch.org/docs/stable/distributed.html#shutdown (function operator())


## PySpark

In [11]:
MODEL_PATH = os.path.abspath("mineru-html")

In [1]:
import pandas as pd
from pyspark.sql.types import *
from pyspark import SparkConf
from pyspark.sql import SparkSession
from pyspark.sql.functions import pandas_udf, col, struct, length, lit, concat
from pyspark.ml.functions import predict_batch_udf

In [2]:
import os
import socket
import datasets
from datasets import load_dataset
datasets.disable_progress_bars()

#### Create Spark Session

In [4]:
conf = SparkConf()

conda_env = os.environ.get("CONDA_PREFIX")
hostname = socket.gethostname()
conf.setMaster(f"spark://{hostname}:7077")
conf.set("spark.pyspark.python", f"{conda_env}/bin/python")
conf.set("spark.pyspark.driver.python", f"{conda_env}/bin/python")
conf.set("spark.executor.cores", "8")
conf.set("spark.task.maxFailures", "1")
conf.set("spark.task.resource.gpu.amount", "0.125")
conf.set("spark.executor.resource.gpu.amount", "1")
conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")
conf.set("spark.python.worker.reuse", "true")

spark = SparkSession.builder.appName("spark-dl-examples").config(conf=conf).getOrCreate()
sc = spark.sparkContext

25/12/09 15:11:02 WARN Utils: Your hostname, cb4ae00-lcedt resolves to a loopback address: 127.0.1.1; using 10.110.47.100 instead (on interface eno1)
25/12/09 15:11:02 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/12/09 15:11:02 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


#### Load and Preprocess DataFrame

Load the WebMainBench [sample dataset](https://github.com/opendatalab/WebMainBench/blob/main/data/sample_dataset.jsonl).

In [5]:
import requests
import json

url = "https://raw.githubusercontent.com/opendatalab/WebMainBench/refs/heads/main/data/sample_dataset.jsonl"
response = requests.get(url)

data = []
for line in response.text.strip().split('\n'):
    if line:
        try:
            item = json.loads(line)
            record = {
                'html': item.get('html'),
                'groundtruth_content': item.get('content') or item.get('convert_main_content') or item.get('groundtruth_content'),
                'url': item.get('url')
            }
            if record['html'] and record['groundtruth_content']:
                data.append(record)
        except json.JSONDecodeError as e:
            print(f"Warning: Failed to parse line: {e}")
            continue

The sample only contains 4 records, so duplicating 25x to get 100 samples.

In [6]:
num_replicas = 25

pdf = pd.DataFrame(data)
pdf = pd.concat([pdf] * num_replicas, ignore_index=True)

print(pdf.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 3 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   html                 100 non-null    object
 1   groundtruth_content  100 non-null    object
 2   url                  100 non-null    object
dtypes: object(3)
memory usage: 2.5+ KB
None


In [21]:
df = spark.createDataFrame(pdf).repartition(2)

df.printSchema()
df.show(5, truncate=80)

root
 |-- html: string (nullable = true)
 |-- groundtruth_content: string (nullable = true)
 |-- url: string (nullable = true)

+--------------------------------------------------------------------------------+----------------------------------------------------------------------------------------------------------+--------------------------------------------------------------------------------+
|                                                                            html|                                                                                       groundtruth_content|                                                                             url|
+--------------------------------------------------------------------------------+----------------------------------------------------------------------------------------------------------+--------------------------------------------------------------------------------+
|<html class="avada-html-layout-wide" lang="en-US" prefix="

25/12/09 15:20:38 WARN TaskSetManager: Stage 15 contains a task of very large size (3883 KiB). The maximum recommended task size is 1000 KiB.


## Inference using Spark DL API

Distributed inference using the PySpark [predict_batch_udf](https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.functions.predict_batch_udf.html#pyspark.ml.functions.predict_batch_udf).

In [23]:
# Launching multiple vLLM v1 engines on the same machine will result in:
# AssertionError: Error in memory profiling. Initial free memory 44.5316162109375 GiB, current free memory 45.44561767578125 GiB.
# This happens when other processes sharing the same container release GPU memory while vLLM is profiling during initialization.
# To fix this, ensure consistent GPU memory allocation or isolate vLLM in its own container.
# https://github.com/vllm-project/vllm/blob/3c680f4a17057d7994af8fbb1dc8c2d98307c890/vllm/v1/worker/gpu_worker.py#L343

In [31]:
def predict_batch_fn():
    import os
    import numpy as np
    from dripper.api import Dripper
    from pyspark import TaskContext

    os.environ["VLLM_USE_V1"] = "0"  # disable v1 to allow concurrent engines
    print(f"Initializing model on worker {TaskContext.get().partitionId()}")
    dripper = Dripper(
        config={
            'model_path': MODEL_PATH,
            'vllm_kwargs': {
                'gpu_memory_utilization': 0.4,
                'max_model_len': 32768,
            }
        }
    )

    def predict(inputs):
        flattened = np.squeeze(inputs).tolist()
        outputs = dripper.process(flattened)
        return np.array([o.main_html for o in outputs])
    
    return predict

In [32]:
generate = predict_batch_udf(predict_batch_fn,
                             return_type=StringType(),
                             batch_size=50)

In [33]:
%%time
# first pass caches model/fn92484
preds = df.withColumn("processed_html", generate(struct("html")))
results = preds.collect()

25/12/09 15:22:41 WARN TaskSetManager: Stage 24 contains a task of very large size (3883 KiB). The maximum recommended task size is 1000 KiB.


CPU times: user 37.7 ms, sys: 35.5 ms, total: 73.3 ms
Wall time: 3min 25s
